# Solar Filament Segmentation 2026 — solarfil v2 (full pipeline)

**Settings (right panel):** Accelerator = GPU (T4 x2 or P100), Internet = ON, Input = competition data.
To continue from an earlier run, also add that run's *output* as input (checkpoints and OOF files are found automatically).

Pipeline: limb-darkening flattening → U-Net (EfficientNet-B4) at 1536² on 768² crops with
multi-annotator soft targets + an instance-border channel → 8x dihedral TTA + fold ensemble →
fragment linking / border splitting / filtering tuned with the **exact official PQ metric** on out-of-fold predictions.

In [ ]:
# ======================= SETTINGS =======================
FOLDS      = [1, 2]
EPOCHS     = 36
ENCODER    = "efficientnet-b4"
SIZE       = 1536
CROP       = 768
BATCH      = 6
RUN_TRAIN  = True
RUN_TUNE   = False         # needs out-of-fold files of ALL 5 folds (current + earlier outputs)
RUN_INFER  = False         # writes /kaggle/working/submission.csv using every checkpoint found
TUNE_PHOTOS = 300          # OOF photos used for post-processing tuning (CPU time ~ linear)
# ========================================================

In [ ]:
!pip install -q segmentation-models-pytorch==0.5.0 pycocotools 2>&1 | tail -1

In [ ]:
import os; os.makedirs('solarfil', exist_ok=True)

In [ ]:
%%writefile solarfil/__init__.py
"""solarfil: instance segmentation of solar filaments in GONG H-alpha images.

Pipeline
--------
1. preprocess  : limb-darkening flattening + contrast channels (preprocess.py)
2. targets     : multi-annotator soft foreground + instance-border maps (annotations.py)
3. model       : U-Net (segmentation_models_pytorch) with 2 outputs: foreground, border (model.py)
4. training    : high-resolution crops, Dice+BCE, EMA, GroupKFold by photo (train.py)
5. inference   : full-disk prediction, dihedral TTA, fold ensemble (inference.py)
6. instances   : threshold -> fragment linking -> border splitting -> hole filling (postprocess.py)
7. evaluation  : exact re-implementation of the official Panoptic Quality metric (metric.py)
"""
__version__ = "2.0.0"

In [ ]:
%%writefile solarfil/annotations.py
"""MAGFiLO COCO annotations -> ground-truth RLEs and soft training targets.

Important dataset facts (verified on the 2026 Kaggle train split):
  * 707 unique photos, 1154 COCO `images`: a photo can be annotated by 1-3 annotators.
    Each annotator copy has its own image id "<annotator>-<photo stem>".
  * The official metric scores the prediction for a photo against EVERY annotator copy
    separately (see metric.py), so the best prediction is a consensus of annotators.
  * Every annotation is a single filled polygon.
"""
from __future__ import annotations

import json
import os
from collections import defaultdict
from dataclasses import dataclass, field

import cv2
import numpy as np
from pycocotools import mask as mask_util

NATIVE = 2048


@dataclass
class Photo:
    stem: str
    file_name: str
    # annotator image id -> list of polygons (flat [x0, y0, x1, y1, ...])
    annotators: dict = field(default_factory=dict)


def load_photos(json_path: str) -> dict[str, Photo]:
    with open(json_path, "r", encoding="utf-8") as f:
        coco = json.load(f)
    anns = defaultdict(list)
    for a in coco["annotations"]:
        anns[a["image_id"]].append(a)
    photos: dict[str, Photo] = {}
    for im in coco["images"]:
        stem = os.path.splitext(im["file_name"])[0]
        p = photos.setdefault(stem, Photo(stem=stem, file_name=im["file_name"]))
        polys = []
        for a in anns.get(im["id"], []):
            seg = a["segmentation"]
            if isinstance(seg, list):
                polys.extend([s for s in seg if len(s) >= 6])
        p.annotators[im["id"]] = polys
    return photos


def poly_to_rle(poly: list, h: int = NATIVE, w: int = NATIVE) -> dict:
    rles = mask_util.frPyObjects([poly], h, w)
    return mask_util.merge(rles)


def photo_gt_rles(photo: Photo) -> dict[str, list[dict]]:
    """annotator id -> list of COCO RLE dicts (one per filament), exactly like annToMask."""
    return {aid: [poly_to_rle(p) for p in polys] for aid, polys in photo.annotators.items()}


def soft_targets(photo: Photo, size: int, border_px: int = 5) -> tuple[np.ndarray, np.ndarray]:
    """Soft foreground and instance-border targets at (size, size), averaged over annotators.

    foreground: fraction of annotators (and of the downsampled pixel area) covered by a filament.
    border    : pixels where dilated masks of two different filaments of the same annotator meet.
                Teaches the network where touching filaments must be split.
    """
    fg = np.zeros((size, size), np.float32)
    bd = np.zeros((size, size), np.float32)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * border_px + 1, 2 * border_px + 1))
    n_ann = max(len(photo.annotators), 1)
    for polys in photo.annotators.values():
        union = np.zeros((NATIVE, NATIVE), np.uint8)
        count = np.zeros((NATIVE, NATIVE), np.uint8)
        for p in polys:
            m = mask_util.decode(poly_to_rle(p))
            union |= m
            if len(polys) > 1:
                count += cv2.dilate(m, k)
        fg += cv2.resize(union.astype(np.float32), (size, size), interpolation=cv2.INTER_AREA)
        if len(polys) > 1:
            b = (count >= 2).astype(np.float32)
            bd += cv2.resize(b, (size, size), interpolation=cv2.INTER_AREA)
    return fg / n_ann, np.clip(bd / n_ann, 0, 1)


def gt_frame(photos: dict[str, Photo], stems: list[str] | None = None):
    """Ground truth in the official submission format (annotator-prefixed filament ids)."""
    import pandas as pd

    rows = []
    for stem in stems or list(photos):
        for aid, polys in photos[stem].annotators.items():
            for i, p in enumerate(polys, 1):
                rle = poly_to_rle(p)
                rows.append({"filament_id": f"{aid}_{i}", "segmentation_rle": rle["counts"].decode()})
    return pd.DataFrame(rows)

In [ ]:
%%writefile solarfil/data.py
"""Caching, folds and PyTorch datasets."""
from __future__ import annotations

import os
import random
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset

from .annotations import Photo, soft_targets
from .preprocess import load_input


# ----------------------------------------------------------------------------- folds
def make_folds(stems: list[str], n_folds: int = 5, seed: int = 42) -> pd.DataFrame:
    """Group by observation date: same-day photos from different GONG sites see the same filaments."""
    df = pd.DataFrame({"stem": sorted(stems)})
    df["group"] = df["stem"].str[:8]
    groups = df["group"].unique().tolist()
    rng = random.Random(seed)
    rng.shuffle(groups)
    gfold = {g: i % n_folds for i, g in enumerate(groups)}
    df["fold"] = df["group"].map(gfold)
    return df


# ----------------------------------------------------------------------------- cache
def _cache_one(args):
    photo, img_path, cache_dir, size, border_px = args
    xp = os.path.join(cache_dir, f"{photo.stem}_x.npy")
    yp = os.path.join(cache_dir, f"{photo.stem}_y.npy")
    if os.path.exists(xp) and os.path.exists(yp):
        return photo.stem
    x = load_input(img_path, size)
    fg, bd = soft_targets(photo, size, border_px=border_px)
    y = np.dstack([np.round(fg * 255), np.round(bd * 255)]).astype(np.uint8)
    np.save(xp, x)
    np.save(yp, y)
    return photo.stem


def build_cache(photos: dict[str, Photo], img_dir: str, cache_dir: str, size: int,
                border_px: int = 5, workers: int = 4):
    os.makedirs(cache_dir, exist_ok=True)
    jobs = [(p, os.path.join(img_dir, p.file_name), cache_dir, size, border_px) for p in photos.values()]
    if workers <= 1:
        for j in jobs:
            _cache_one(j)
    else:
        with ProcessPoolExecutor(workers) as ex:
            for i, _ in enumerate(ex.map(_cache_one, jobs, chunksize=4)):
                if i % 100 == 0:
                    print(f"cache {i}/{len(jobs)}", flush=True)


def build_test_cache(paths: list[str], cache_dir: str, size: int, workers: int = 4):
    os.makedirs(cache_dir, exist_ok=True)

    todo = [p for p in paths
            if not os.path.exists(os.path.join(cache_dir, os.path.splitext(os.path.basename(p))[0] + "_x.npy"))]
    with ProcessPoolExecutor(max(workers, 1)) as ex:
        for p, x in zip(todo, ex.map(load_input, todo, [size] * len(todo))):
            np.save(os.path.join(cache_dir, os.path.splitext(os.path.basename(p))[0] + "_x.npy"), x)


# ----------------------------------------------------------------------------- augmentation
def dihedral(a: np.ndarray, k: int) -> np.ndarray:
    """k in 0..7: rot90 by k%4, then horizontal flip if k>=4."""
    a = np.rot90(a, k % 4, axes=(0, 1))
    if k >= 4:
        a = a[:, ::-1]
    return a


def photometric(x: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    x = x.astype(np.float32) / 255.0
    if rng.random() < 0.8:  # contrast / brightness / gamma, per channel jitter
        c = rng.uniform(0.75, 1.3)
        b = rng.uniform(-0.08, 0.08)
        g = rng.uniform(0.75, 1.35)
        x = np.clip((x - 0.5) * c + 0.5 + b, 0, 1) ** g
    if rng.random() < 0.3:
        x = x + rng.normal(0, rng.uniform(0.005, 0.03), x.shape).astype(np.float32)
    if rng.random() < 0.2:
        import cv2
        x = cv2.GaussianBlur(x, (0, 0), rng.uniform(0.5, 1.2))
    return np.clip(x, 0, 1)


# ----------------------------------------------------------------------------- datasets
class CropDataset(Dataset):
    """Random crops from cached full-disk arrays; `pos_frac` of crops are centred on a filament."""

    def __init__(self, stems, cache_dir, crop=768, samples_per_epoch=None, pos_frac=0.7, seed=0):
        self.stems = list(stems)
        self.cache_dir = cache_dir
        self.crop = crop
        self.n = samples_per_epoch or 2 * len(self.stems)
        self.pos_frac = pos_frac
        self.seed = seed
        self._fgidx = {}

    def __len__(self):
        return self.n

    def _arrays(self, stem):
        x = np.load(os.path.join(self.cache_dir, f"{stem}_x.npy"), mmap_mode="r")
        y = np.load(os.path.join(self.cache_dir, f"{stem}_y.npy"), mmap_mode="r")
        return x, y

    def __getitem__(self, i):
        rng = np.random.default_rng((self.seed * 1_000_003 + i + random.getrandbits(31)) % 2**32)
        stem = self.stems[rng.integers(len(self.stems))]
        x, y = self._arrays(stem)
        S, c = x.shape[0], self.crop
        if rng.random() < self.pos_frac:
            if stem not in self._fgidx:
                ys, xs = np.nonzero(np.asarray(y[::8, ::8, 0]) > 64)
                self._fgidx[stem] = (ys * 8, xs * 8)
            ys, xs = self._fgidx[stem]
            if len(ys):
                j = rng.integers(len(ys))
                cy = int(ys[j] + rng.integers(-c // 3, c // 3 + 1))
                cx = int(xs[j] + rng.integers(-c // 3, c // 3 + 1))
            else:
                cy, cx = rng.integers(c // 2, S - c // 2 + 1, size=2)
        else:
            cy, cx = rng.integers(c // 2, S - c // 2 + 1, size=2)
        y0 = int(np.clip(cy - c // 2, 0, S - c))
        x0 = int(np.clip(cx - c // 2, 0, S - c))
        xc = np.asarray(x[y0:y0 + c, x0:x0 + c])
        yc = np.asarray(y[y0:y0 + c, x0:x0 + c]).astype(np.float32) / 255.0
        k = int(rng.integers(8))
        xc, yc = dihedral(xc, k), dihedral(yc, k)
        xc = photometric(np.ascontiguousarray(xc), rng)
        return (torch.from_numpy(np.ascontiguousarray(xc.transpose(2, 0, 1))),
                torch.from_numpy(np.ascontiguousarray(yc.transpose(2, 0, 1))))


def load_full(stem: str, cache_dir: str) -> np.ndarray:
    """Full cached input as float32 CHW tensor in [0,1]."""
    x = np.load(os.path.join(cache_dir, f"{stem}_x.npy"))
    return torch.from_numpy(x.transpose(2, 0, 1).astype(np.float32) / 255.0)

In [ ]:
%%writefile solarfil/infer.py
"""Predict the test set and write submission.csv.

python -m solarfil.infer --data_root <root> --ckpts work/*.pth --postcfg postcfg.json --out submission.csv
"""
from __future__ import annotations

import argparse
import glob
import json
import os
import time

import numpy as np
import pandas as pd
import torch
from pycocotools import mask as mask_util

from .data import build_test_cache, load_full
from .model import FilamentNet, predict_full
from .postprocess import PostCfg, instances_rle


def load_models(paths, dev):
    groups = {}
    for p in paths:
        ck = torch.load(p, map_location="cpu", weights_only=False)
        a = ck["args"]
        m = FilamentNet(a["arch"], a["encoder"], weights=None)
        m.load_state_dict(ck["state_dict"])
        groups.setdefault(a["size"], []).append(m.to(dev).eval())
        print(f"loaded {os.path.basename(p)} (val PQ {ck.get('pq', float('nan')):.4f}, size {a['size']})")
    return groups


def check_submission(df: pd.DataFrame):
    assert list(df.columns) == ["filament_id", "segmentation_rle"]
    assert df["filament_id"].is_unique
    assert not df.isnull().any().any()
    for stem, g in df.groupby(df["filament_id"].str.split("_", n=1).str[0]):
        rles = [{"size": [2048, 2048], "counts": r.encode()} for r in g["segmentation_rle"]]
        if len(rles) > 1:
            inter = mask_util.area(mask_util.merge(rles, intersect=False))
            assert inter == mask_util.area(rles).sum(), f"overlapping masks in {stem}"


def main(argv=None):
    a = argparse.ArgumentParser()
    a.add_argument("--data_root", required=True)
    a.add_argument("--ckpts", nargs="+", required=True)
    a.add_argument("--postcfg", default="")
    a.add_argument("--work", default="work")
    a.add_argument("--tta", type=int, default=8)
    a.add_argument("--out", default="submission.csv")
    a.add_argument("--save_probs", default="")
    args = a.parse_args(argv)

    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    cfg = PostCfg(**json.load(open(args.postcfg))["cfg"]) if args.postcfg else PostCfg()
    print("post-processing:", cfg)
    paths = sorted(glob.glob(os.path.join(args.data_root, "test", "test_images", "*.jp*g")))
    groups = load_models(args.ckpts, dev)
    for size in groups:
        build_test_cache(paths, os.path.join(args.work, f"test_cache_{size}"), size)

    rows, t0 = [], time.time()
    for path in paths:
        stem = os.path.splitext(os.path.basename(path))[0]
        probs, top = [], max(groups)
        for size, models in groups.items():
            p = predict_full(models, load_full(stem, os.path.join(args.work, f"test_cache_{size}")), tta=args.tta)
            if size != top:  # models trained at different resolutions: average at the largest one
                p = torch.nn.functional.interpolate(p[None], size=(top, top), mode="bilinear", align_corners=False)[0]
            probs.append((p.numpy(), len(models)))
        p = sum(q * n for q, n in probs) / sum(n for _, n in probs)
        if args.save_probs:
            os.makedirs(args.save_probs, exist_ok=True)
            np.savez_compressed(os.path.join(args.save_probs, f"{stem}.npz"), p=np.round(p * 255).astype(np.uint8))
        for i, r in enumerate(instances_rle(p[0], p[1], cfg), 1):
            rows.append({"filament_id": f"{stem}_{i}", "segmentation_rle": r})
    df = pd.DataFrame(rows, columns=["filament_id", "segmentation_rle"])
    check_submission(df)
    df.to_csv(args.out, index=False)
    n_img = df["filament_id"].str.split("_", n=1).str[0].nunique()
    print(f"wrote {args.out}: {len(df)} filaments in {n_img}/{len(paths)} images ({time.time() - t0:.0f}s)")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile solarfil/metric.py
"""Exact re-implementation of the official Panoptic Quality metric (self-evaluation notebook).

Official behaviour reproduced here:
  * GT rows are keyed "<annotator>-<stem>_<k>", prediction rows "<stem>_<k>".
  * For every annotator copy of a photo, ALL predictions of that photo are compared with
    that annotator's filaments; TP iff IoU > 0.5 (strict), unmatched preds = FP,
    unmatched GT = FN.  PQ = sum(IoU_TP) / (TP + FP/2 + FN/2), pooled over everything.
  * Photos without GT (e.g. the other half of the test set) are ignored.
IoUs are computed on RLEs with pycocotools (identical numbers, ~100x faster than dense
2048x2048 matmuls).
"""
from __future__ import annotations

from collections import defaultdict
from dataclasses import dataclass

import numpy as np
from pycocotools import mask as mask_util

SIZE = [2048, 2048]


def _as_rle(r):
    if isinstance(r, dict):
        return r
    return {"size": SIZE, "counts": r.encode() if isinstance(r, str) else r}


@dataclass
class PQStats:
    iou_sum: float = 0.0
    tp: int = 0
    fp: int = 0
    fn: int = 0

    def add(self, o: "PQStats"):
        self.iou_sum += o.iou_sum
        self.tp += o.tp
        self.fp += o.fp
        self.fn += o.fn
        return self

    @property
    def pq(self) -> float:
        d = self.tp + 0.5 * self.fp + 0.5 * self.fn
        return self.iou_sum / d if d > 0 else 0.0

    @property
    def sq(self) -> float:
        return self.iou_sum / self.tp if self.tp else 0.0

    @property
    def dq(self) -> float:
        d = self.tp + 0.5 * self.fp + 0.5 * self.fn
        return self.tp / d if d > 0 else 0.0

    def __repr__(self):
        return (f"PQ={self.pq:.4f} (SQ={self.sq:.4f} DQ={self.dq:.4f}) "
                f"TP={self.tp} FP={self.fp} FN={self.fn}")


def image_stats(gt_rles: list, pred_rles: list) -> PQStats:
    """Stats for one annotator copy vs. the predictions of that photo."""
    n_gt, n_pred = len(gt_rles), len(pred_rles)
    if n_gt == 0:
        return PQStats(fp=n_pred)
    if n_pred == 0:
        return PQStats(fn=n_gt)
    g = [_as_rle(r) for r in gt_rles]
    p = [_as_rle(r) for r in pred_rles]
    iou = np.asarray(mask_util.iou(p, g, [0] * n_gt))  # (n_pred, n_gt)
    hit = iou > 0.5
    return PQStats(
        iou_sum=float(iou[hit].sum()),
        tp=int(hit.sum()),
        fp=int((hit.sum(axis=1) == 0).sum()),
        fn=int((hit.sum(axis=0) == 0).sum()),
    )


def photo_stats(gt_by_annotator: dict, pred_rles: list) -> PQStats:
    s = PQStats()
    for rles in gt_by_annotator.values():
        if len(rles):
            s.add(image_stats(rles, pred_rles))
    return s


def score_frames(gt_df, pred_df) -> PQStats:
    """Score two DataFrames in the official submission format."""
    gt = defaultdict(lambda: defaultdict(list))
    for fid, rle in zip(gt_df["filament_id"], gt_df["segmentation_rle"]):
        ann_img = fid.split("_", 1)[0]
        stem = ann_img.split("-", 1)[1]
        gt[stem][ann_img].append(rle)
    pred = defaultdict(list)
    for fid, rle in zip(pred_df["filament_id"], pred_df["segmentation_rle"]):
        pred[fid.split("_", 1)[0]].append(rle)
    total = PQStats()
    for stem, anns in gt.items():
        total.add(photo_stats(anns, pred.get(stem, [])))
    return total

In [ ]:
%%writefile solarfil/model.py
"""Network, losses and (TTA) inference."""
from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F
import segmentation_models_pytorch as smp

MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


class FilamentNet(nn.Module):
    """U-Net with 2 output channels: 0 = filament foreground, 1 = instance border."""

    def __init__(self, arch="Unet", encoder="efficientnet-b4", weights="imagenet"):
        super().__init__()
        kw = dict(encoder_name=encoder, encoder_weights=weights, in_channels=3, classes=2)
        if arch in ("Unet", "UnetPlusPlus"):
            kw["decoder_attention_type"] = "scse"
        self.net = getattr(smp, arch)(**kw)
        self.register_buffer("mean", MEAN.clone(), persistent=False)
        self.register_buffer("std", STD.clone(), persistent=False)

    def forward(self, x):  # x in [0,1]
        return self.net((x - self.mean) / self.std)


def soft_dice_loss(logits, target, eps=1.0):
    p = torch.sigmoid(logits)
    inter = (p * target).sum(dim=(1, 2))
    den = p.sum(dim=(1, 2)) + target.sum(dim=(1, 2))
    return (1 - (2 * inter + eps) / (den + eps)).mean()


class FilamentLoss(nn.Module):
    def __init__(self, border_weight=0.5, pos_weight=3.0):
        super().__init__()
        self.bw = border_weight
        self.pw = pos_weight

    def forward(self, logits, target):
        fg_l, bd_l = logits[:, 0], logits[:, 1]
        fg_t, bd_t = target[:, 0], target[:, 1]
        pw = torch.tensor(self.pw, device=logits.device)
        loss = F.binary_cross_entropy_with_logits(fg_l, fg_t, pos_weight=pw) + soft_dice_loss(fg_l, fg_t)
        if self.bw > 0:
            loss = loss + self.bw * (F.binary_cross_entropy_with_logits(bd_l, bd_t, pos_weight=pw * 3)
                                     + 0.5 * soft_dice_loss(bd_l, bd_t))
        return loss


@torch.no_grad()
def predict_full(models, x: torch.Tensor, tta: int = 8, amp: bool = True) -> torch.Tensor:
    """x: (3,H,W) in [0,1]. Returns averaged sigmoid probs (2,H,W) on CPU.

    tta: number of dihedral transforms (1, 2, 4 or 8)."""
    if not isinstance(models, (list, tuple)):
        models = [models]
    dev = next(models[0].parameters()).device
    xb = x.unsqueeze(0).to(dev)
    acc = None
    n = 0
    for k in range(tta):
        r, f = k % 4, k >= 4
        xt = torch.rot90(xb, r, dims=(2, 3))
        if f:
            xt = torch.flip(xt, dims=(3,))
        for m in models:
            with torch.autocast(device_type=dev.type, enabled=amp and dev.type == "cuda"):
                p = torch.sigmoid(m(xt).float())
            if f:
                p = torch.flip(p, dims=(3,))
            p = torch.rot90(p, -r, dims=(2, 3))
            acc = p if acc is None else acc + p
            n += 1
    return (acc / n)[0].float().cpu()


class EMA:
    def __init__(self, model, decay=0.999):
        import copy
        self.decay = decay
        self.ema = copy.deepcopy(model).eval()
        for p in self.ema.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        msd = model.state_dict()
        for k, v in self.ema.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(self.decay).add_(msd[k].detach(), alpha=1 - self.decay)
            else:
                v.copy_(msd[k])

In [ ]:
%%writefile solarfil/postprocess.py
"""Probability maps -> non-overlapping filament instances (COCO RLE at 2048x2048).

Panoptic Quality punishes two failure modes that plain "threshold + connected components"
produces on filaments:
  * fragmentation: one long faint filament breaks into several blobs -> every blob is an FP
    and the real filament an FN.            -> fix: link fragments closer than `link_px`.
  * over-merging : two neighbouring filaments touch -> one blob, IoU < 0.5 with both.
                                             -> fix: cut along the predicted border map.
Annotators draw filled polygons, so instance holes are filled.
All parameters are tuned on out-of-fold predictions with the official metric (tune.py).
"""
from __future__ import annotations

from dataclasses import asdict, dataclass

import cv2
import numpy as np
from pycocotools import mask as mask_util
from scipy import ndimage as ndi

NATIVE = 2048


@dataclass
class PostCfg:
    thr: float = 0.45          # foreground probability threshold
    close_px: int = 0          # morphological closing radius before labelling
    min_cc: int = 40           # drop tiny specks before linking (native px)
    link_px: int = 8           # fragments closer than this are one filament (native px)
    border_thr: float = 1.01   # >1 disables border splitting
    min_area: int = 250        # final instance area filter (native px)
    min_score: float = 0.50    # mean foreground prob inside instance
    fill_holes: bool = True

    def as_dict(self):
        return asdict(self)


def _disk(r: int):
    return cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * r + 1, 2 * r + 1))


def upsample(p: np.ndarray, size: int = NATIVE) -> np.ndarray:
    p = np.ascontiguousarray(p, dtype=np.float32)
    if p.shape[0] == size:
        return p
    return cv2.resize(p, (size, size), interpolation=cv2.INTER_LINEAR)


def instances(fg: np.ndarray, border: np.ndarray | None, cfg: PostCfg) -> list[np.ndarray]:
    """fg/border: float maps in [0,1] at any square resolution. Returns list of uint8 masks (2048²)."""
    fg = upsample(fg)
    binary = (fg > cfg.thr).astype(np.uint8)
    if cfg.close_px > 0:
        binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, _disk(cfg.close_px))
    if not binary.any():
        return []

    # 1) remove specks
    n, lab, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    keep = np.zeros(n, bool)
    keep[1:] = stats[1:, cv2.CC_STAT_AREA] >= cfg.min_cc
    binary = keep[lab].astype(np.uint8)
    if not binary.any():
        return []

    # 2) link nearby fragments, 3) cut along predicted borders
    core = cv2.dilate(binary, _disk(cfg.link_px // 2)) if cfg.link_px > 1 else binary.copy()
    use_border = border is not None and cfg.border_thr <= 1.0
    if use_border:
        cut = upsample(border) > cfg.border_thr
        core[cut] = 0
    n, lab = cv2.connectedComponents(core, connectivity=8)
    lab = lab.astype(np.int32)
    lab[binary == 0] = 0
    if use_border:
        # border pixels that belong to the foreground: give them to the nearest instance
        orphan = (binary > 0) & (lab == 0)
        if orphan.any() and n > 1:
            _, (iy, ix) = ndi.distance_transform_edt(lab == 0, return_indices=True)
            lab[orphan] = lab[iy[orphan], ix[orphan]]

    # 4) per-instance filtering
    out = []
    for i, sl in enumerate(ndi.find_objects(lab), 1):
        if sl is None:
            continue
        m = lab[sl] == i
        if cfg.fill_holes:
            m = ndi.binary_fill_holes(m)
            m &= (lab[sl] == i) | (lab[sl] == 0)  # never steal pixels of another instance
        area = int(m.sum())
        if area < cfg.min_area:
            continue
        if float(fg[sl][m].mean()) < cfg.min_score:
            continue
        full = np.zeros((NATIVE, NATIVE), np.uint8)
        full[sl][m] = 1
        out.append(full)
    return out


def encode(masks: list[np.ndarray]) -> list[str]:
    return [mask_util.encode(np.asfortranarray(m))["counts"].decode() for m in masks]


def instances_rle(fg, border, cfg: PostCfg) -> list[str]:
    return encode(instances(fg, border, cfg))

In [ ]:
%%writefile solarfil/preprocess.py
"""Image preprocessing for GONG H-alpha full-disk observations.

Filaments are dark, elongated absorption features on the solar disk. Their contrast
is low and the disk brightness falls off towards the limb (limb darkening) and varies
between the six GONG sites. We therefore build a 3-channel input:

  ch0  global intensity (percentile-normalised)
  ch1  flattened image  = image / smooth background  (removes limb darkening / gradients)
  ch2  local contrast    = CLAHE of the flattened image

All channels are uint8 so they can be cached cheaply.
"""
from __future__ import annotations

import cv2
import numpy as np

NATIVE = 2048


def read_gray(path: str) -> np.ndarray:
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(path)
    return img


def disk_mask(img: np.ndarray) -> np.ndarray:
    """Binary mask of the solar disk (Otsu on a blurred image, largest component, filled)."""
    small = cv2.resize(img, (512, 512), interpolation=cv2.INTER_AREA)
    small = cv2.GaussianBlur(small, (0, 0), 3)
    _, th = cv2.threshold(small, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    n, lab, stats, _ = cv2.connectedComponentsWithStats(th)
    if n > 1:
        k = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
        th = (lab == k).astype(np.uint8) * 255
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    filled = np.zeros_like(th)
    if cnts:
        cv2.drawContours(filled, cnts, -1, 255, -1)
    filled = cv2.erode(filled, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)))
    return cv2.resize(filled, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST) > 0


def flatten(img: np.ndarray, mask: np.ndarray, sigma: float = 24.0) -> np.ndarray:
    """Divide by a smooth background estimated inside the disk (normalised convolution)."""
    s = 4  # work at 1/4 resolution for speed; the background is smooth anyway
    h, w = img.shape
    small = cv2.resize(img.astype(np.float32), (w // s, h // s), interpolation=cv2.INTER_AREA)
    m = cv2.resize(mask.astype(np.float32), (w // s, h // s), interpolation=cv2.INTER_AREA)
    # median filter first so dark filaments don't pull the background down
    med = cv2.medianBlur(np.clip(small, 0, 255).astype(np.uint8), 15).astype(np.float32)
    num = cv2.GaussianBlur(med * m, (0, 0), sigma / s * 2)
    den = cv2.GaussianBlur(m, (0, 0), sigma / s * 2)
    bg = num / np.maximum(den, 1e-3)
    bg = cv2.resize(bg, (w, h), interpolation=cv2.INTER_LINEAR)
    ratio = img.astype(np.float32) / np.maximum(bg, 1.0)
    ratio[~mask] = 1.0
    # map ratio 0.6..1.25 -> 0..255
    out = (ratio - 0.6) / (1.25 - 0.6) * 255.0
    return np.clip(out, 0, 255).astype(np.uint8)


def make_input(img: np.ndarray) -> np.ndarray:
    """Return (H, W, 3) uint8 network input at native resolution."""
    mask = disk_mask(img)
    vals = img[mask] if mask.any() else img.ravel()
    lo, hi = np.percentile(vals, [0.5, 99.5])
    ch0 = np.clip((img.astype(np.float32) - lo) / max(hi - lo, 1.0) * 255.0, 0, 255).astype(np.uint8)
    ch1 = flatten(img, mask)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(16, 16))
    ch2 = clahe.apply(ch1)
    ch2[~mask] = 0
    return np.dstack([ch0, ch1, ch2])


def load_input(path: str, size: int) -> np.ndarray:
    """Read an observation and return the 3-channel input resized to (size, size)."""
    x = make_input(read_gray(path))
    if size != x.shape[0]:
        x = cv2.resize(x, (size, size), interpolation=cv2.INTER_AREA)
    return x

In [ ]:
%%writefile solarfil/train.py
"""Train one fold.

python -m solarfil.train --data_root <MAGFiLO_1.0_Kaggle_2026> --work /kaggle/working --fold 0
"""
from __future__ import annotations

import argparse
import json
import math
import os
import time

import numpy as np
import torch
from torch.utils.data import DataLoader

from .annotations import load_photos, photo_gt_rles
from .data import CropDataset, build_cache, load_full, make_folds
from .metric import PQStats, photo_stats
from .model import EMA, FilamentLoss, FilamentNet, predict_full
from .postprocess import PostCfg, instances_rle


def get_args(argv=None):
    a = argparse.ArgumentParser()
    a.add_argument("--data_root", required=True)
    a.add_argument("--work", default="work")
    a.add_argument("--fold", type=int, default=0)
    a.add_argument("--n_folds", type=int, default=5)
    a.add_argument("--size", type=int, default=1536, help="full-disk working resolution")
    a.add_argument("--crop", type=int, default=768)
    a.add_argument("--arch", default="Unet")
    a.add_argument("--encoder", default="efficientnet-b4")
    a.add_argument("--weights", default="imagenet")
    a.add_argument("--epochs", type=int, default=40)
    a.add_argument("--bs", type=int, default=6)
    a.add_argument("--lr", type=float, default=6e-4)
    a.add_argument("--wd", type=float, default=1e-4)
    a.add_argument("--samples_per_epoch", type=int, default=0, help="0 -> 3 crops per training photo")
    a.add_argument("--border_weight", type=float, default=0.5)
    a.add_argument("--val_every", type=int, default=4)
    a.add_argument("--val_tta", type=int, default=2)
    a.add_argument("--workers", type=int, default=4)
    a.add_argument("--limit", type=int, default=0, help="debug: use only N photos")
    a.add_argument("--ema", type=float, default=0.998)
    a.add_argument("--seed", type=int, default=42)
    return a.parse_args(argv)


def evaluate(model, stems, cache_dir, gts, cfg: PostCfg, tta: int) -> PQStats:
    total = PQStats()
    for s in stems:
        p = predict_full(model, load_full(s, cache_dir), tta=tta).numpy()
        total.add(photo_stats(gts[s], instances_rle(p[0], p[1], cfg)))
    return total


def main(argv=None):
    args = get_args(argv)
    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    json_path = os.path.join(args.data_root, "train", "MAGFiLO_1.0_Annotations_kaggle2026_train.json")
    img_dir = os.path.join(args.data_root, "train", "train_images")
    photos = load_photos(json_path)
    photos = {k: v for k, v in photos.items() if os.path.exists(os.path.join(img_dir, v.file_name))}
    if args.limit:
        photos = dict(list(photos.items())[: args.limit])
    cache_dir = os.path.join(args.work, f"cache_{args.size}")
    print(f"photos={len(photos)} device={dev} cache={cache_dir}", flush=True)
    build_cache(photos, img_dir, cache_dir, args.size, workers=args.workers)

    folds = make_folds(list(photos), args.n_folds, args.seed)
    folds.to_csv(os.path.join(args.work, "folds.csv"), index=False)
    tr = folds.loc[folds.fold != args.fold, "stem"].tolist()
    va = folds.loc[folds.fold == args.fold, "stem"].tolist()
    gts = {s: photo_gt_rles(photos[s]) for s in va}
    print(f"fold {args.fold}: train={len(tr)} val={len(va)}", flush=True)

    spe = args.samples_per_epoch or 3 * len(tr)
    ds = CropDataset(tr, cache_dir, crop=args.crop, samples_per_epoch=spe, seed=args.seed)
    dl = DataLoader(ds, batch_size=args.bs, shuffle=True, num_workers=args.workers, drop_last=True,
                    pin_memory=dev.type == "cuda", persistent_workers=args.workers > 0)

    model = FilamentNet(args.arch, args.encoder, None if args.weights == "none" else args.weights).to(dev)
    ema = EMA(model, decay=args.ema)
    crit = FilamentLoss(border_weight=args.border_weight)
    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.wd)
    steps = args.epochs * len(dl)
    warm = min(300, steps // 10)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, steps - warm))) * 0.98 + 0.02)
    scaler = torch.amp.GradScaler("cuda", enabled=dev.type == "cuda")

    tag = f"{args.arch}_{args.encoder}_s{args.size}_f{args.fold}"
    ckpt = os.path.join(args.work, f"{tag}.pth")
    best, log = -1.0, []
    cfg = PostCfg()
    for ep in range(1, args.epochs + 1):
        model.train()
        t0, tot = time.time(), 0.0
        for x, y in dl:
            x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)
            with torch.autocast(device_type=dev.type, enabled=dev.type == "cuda"):
                out = model(x)
            loss = crit(out.float(), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            ema.update(model)
            tot += loss.item()
        msg = f"ep {ep:02d} loss {tot / len(dl):.4f} lr {sched.get_last_lr()[0]:.2e} {time.time() - t0:.0f}s"
        if ep % args.val_every == 0 or ep == args.epochs:
            st = evaluate(ema.ema, va, cache_dir, gts, cfg, args.val_tta)
            msg += f" | val {st}"
            log.append({"epoch": ep, "pq": st.pq, "sq": st.sq, "dq": st.dq})
            if st.pq > best:
                best = st.pq
                torch.save({"state_dict": ema.ema.state_dict(), "args": vars(args), "epoch": ep, "pq": best}, ckpt)
                msg += " *"
        print(msg, flush=True)

    # out-of-fold probabilities of the best checkpoint (for post-processing tuning)
    sd = torch.load(ckpt, map_location=dev)["state_dict"]
    model.load_state_dict(sd)
    model.eval()
    oof_dir = os.path.join(args.work, f"oof_{args.arch}_{args.encoder}_s{args.size}")
    os.makedirs(oof_dir, exist_ok=True)
    for s in va:
        p = predict_full(model, load_full(s, cache_dir), tta=8).numpy()
        np.savez_compressed(os.path.join(oof_dir, f"{s}.npz"), p=np.round(p * 255).astype(np.uint8))
    with open(os.path.join(args.work, f"{tag}_log.json"), "w") as f:
        json.dump({"best_pq": best, "log": log, "args": vars(args)}, f, indent=1)
    print(f"done fold {args.fold}: best val PQ {best:.4f} -> {ckpt}", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile solarfil/tune.py
"""Tune post-processing on out-of-fold probabilities with the official PQ metric.

python -m solarfil.tune --data_root <root> --oof work/oof_Unet_efficientnet-b4_s1536 [--oof other_dir ...]
Multiple --oof dirs are averaged (model ensemble).
"""
from __future__ import annotations

import argparse
import glob
import json
import os
from concurrent.futures import ProcessPoolExecutor
from dataclasses import replace

import numpy as np

from .annotations import load_photos, photo_gt_rles
from .metric import PQStats, photo_stats
from .postprocess import PostCfg, instances_rle

_G = {}


def _load(stem):
    ps = [np.load(os.path.join(d, f"{stem}.npz"))["p"].astype(np.float32) / 255.0 for d in _G["dirs"]]
    return np.mean(ps, axis=0)


def _score_one(args):
    stem, cfg = args
    p = _load(stem)
    return photo_stats(_G["gts"][stem], instances_rle(p[0], p[1], cfg))


def score(stems, cfg, pool) -> PQStats:
    tot = PQStats()
    for s in pool.map(_score_one, [(st, cfg) for st in stems], chunksize=2):
        tot.add(s)
    return tot


GRID = {
    "thr": [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60],
    "link_px": [0, 4, 8, 14, 22, 32],
    "border_thr": [1.01, 0.7, 0.5, 0.35],
    "min_area": [100, 200, 300, 450, 650],
    "min_score": [0.40, 0.50, 0.60, 0.70],
    "close_px": [0, 2, 4],
    "min_cc": [10, 40, 100],
    "fill_holes": [True, False],
}


def main(argv=None):
    a = argparse.ArgumentParser()
    a.add_argument("--data_root", required=True)
    a.add_argument("--oof", nargs="+", required=True)
    a.add_argument("--max_photos", type=int, default=0)
    a.add_argument("--rounds", type=int, default=2)
    a.add_argument("--workers", type=int, default=4)
    a.add_argument("--out", default="postcfg.json")
    args = a.parse_args(argv)

    photos = load_photos(os.path.join(args.data_root, "train", "MAGFiLO_1.0_Annotations_kaggle2026_train.json"))
    stems = sorted(set.intersection(*[{os.path.basename(f)[:-4] for f in glob.glob(os.path.join(d, "*.npz"))}
                                      for d in args.oof]))
    if args.max_photos:
        rng = np.random.default_rng(0)
        stems = sorted(rng.choice(stems, size=min(args.max_photos, len(stems)), replace=False).tolist())
    _G["dirs"] = args.oof
    _G["gts"] = {s: photo_gt_rles(photos[s]) for s in stems}
    print(f"tuning on {len(stems)} OOF photos", flush=True)

    cfg = PostCfg()
    with ProcessPoolExecutor(args.workers, initializer=_G.update, initargs=(dict(_G),)) as pool:
        best = score(stems, cfg, pool)
        print("start", cfg, best, flush=True)
        for r in range(args.rounds):
            for k, vals in GRID.items():
                for v in vals:
                    if getattr(cfg, k) == v:
                        continue
                    c = replace(cfg, **{k: v})
                    st = score(stems, c, pool)
                    flag = ""
                    if st.pq > best.pq + 1e-5:
                        cfg, best, flag = c, st, "  <-- best"
                    print(f"r{r} {k}={v}: {st}{flag}", flush=True)
            print(f"after round {r}: {cfg} {best}", flush=True)
    with open(args.out, "w") as f:
        json.dump({"cfg": cfg.as_dict(), "pq": best.pq, "sq": best.sq, "dq": best.dq,
                   "n_photos": len(stems), "oof": args.oof}, f, indent=1)
    print("saved", args.out)


if __name__ == "__main__":
    main()

In [ ]:
import glob, os, shutil, subprocess, sys, json
DATA = os.path.dirname(os.path.dirname(glob.glob('/kaggle/input/**/MAGFiLO_1.0_Annotations_kaggle2026_train.json', recursive=True)[0]))
WORK = '/kaggle/working'
print('data root:', DATA)
# bring in checkpoints / OOF from earlier sessions (attached as inputs)
for p in glob.glob('/kaggle/input/**/*.pth', recursive=True):
    if not os.path.exists(os.path.join(WORK, os.path.basename(p))):
        shutil.copy(p, WORK); print('reuse', os.path.basename(p))
for d in glob.glob('/kaggle/input/**/oof_*', recursive=True):
    if os.path.isdir(d):
        dst = os.path.join(WORK, os.path.basename(d)); os.makedirs(dst, exist_ok=True)
        for f in glob.glob(d + '/*.npz'):
            if not os.path.exists(os.path.join(dst, os.path.basename(f))): shutil.copy(f, dst)
def run(cmd):
    print('$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True)
    assert r.returncode == 0

In [ ]:
if RUN_TRAIN:
    for f in FOLDS:
        run(f"{sys.executable} -m solarfil.train --data_root {DATA} --work {WORK} --fold {f} --epochs {EPOCHS} "
            f"--encoder {ENCODER} --size {SIZE} --crop {CROP} --bs {BATCH} --workers 4")

In [ ]:
OOF = f"{WORK}/oof_Unet_{ENCODER}_s{SIZE}"
if RUN_TUNE:
    print(len(glob.glob(OOF + '/*.npz')), 'OOF photos available')
    run(f"{sys.executable} -m solarfil.tune --data_root {DATA} --oof {OOF} --max_photos {TUNE_PHOTOS} --workers 4 --out {WORK}/postcfg.json")

In [ ]:
if RUN_INFER:
    ck = ' '.join(sorted(glob.glob(f'{WORK}/Unet_*.pth')))
    pc = f'--postcfg {WORK}/postcfg.json' if os.path.exists(f'{WORK}/postcfg.json') else ''
    run(f"{sys.executable} -m solarfil.infer --data_root {DATA} --ckpts {ck} {pc} --work {WORK} --out {WORK}/submission.csv")
# keep the output small: drop caches
for d in glob.glob(f'{WORK}/cache_*') + glob.glob(f'{WORK}/test_cache_*'):
    shutil.rmtree(d, ignore_errors=True)